# TP-E — Analyse d'avis de médicaments (NLP / sentiment)

**Starter notebook.** Vérifie que les données se chargent, sous-échantillonne à 20 000 avis, donne un premier aperçu. Sujet complet dans `TP_Note_Sujets.md` (§ TP-E) et grille dans `BAREME.md`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
N_SAMPLE = 20_000
pd.set_option('display.max_colwidth', 200)


## 1. Chargement

Le dataset Kaggle est livré sous deux fichiers (`drugsComTrain_raw.csv` + `drugsComTest_raw.csv`). On charge le train. Si tu as une autre version (TSV ou nommage différent), adapte le `read_csv` ci-dessous.


In [ ]:
from pathlib import Path

DATA_PATH = Path('data/drugsComTrain_raw.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_PATH.resolve()}\n"
        "Télécharge depuis :\n"
        "  https://www.kaggle.com/datasets/jessicali9530/kuc-hackathon-winter-2018\n"
        "et place le CSV dans le sous-dossier data/ de ce TP."
    )

# La version Kaggle utilise '\t' comme séparateur dans les fichiers '_raw' — adapter si besoin.
try:
    df = pd.read_csv(DATA_PATH)
    if df.shape[1] == 1:
        raise ValueError('une seule colonne détectée — probablement séparé par tab')
except Exception:
    df = pd.read_csv(DATA_PATH, sep='\t')

print(f'Shape brut : {df.shape}')
df.head(3)


## 2. Sous-échantillonnage à 20 000 avis (stratifié sur le rating)


In [ ]:
df_small = (
    df.groupby('rating', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), max(1, int(N_SAMPLE * len(g) / len(df)))), random_state=RANDOM_STATE))
      .reset_index(drop=True)
)
print(f'Shape après sous-échantillonnage : {df_small.shape}')
print(df_small['rating'].value_counts().sort_index())


## 3. Sanity check — texte et colonnes


In [ ]:
print('--- colonnes ---')
print(df_small.columns.tolist())
print('\n--- valeurs manquantes ---')
print(df_small.isna().sum())
print('\n--- exemples d\'avis ---')
for _, row in df_small.sample(3, random_state=RANDOM_STATE).iterrows():
    print(f'[rating={row["rating"]}] {row["review"][:200]}...')
    print()


## 4. Construction de la cible binaire (positif / négatif)

Convention du sujet : `positif` si rating ≥ 7, `négatif` si rating ≤ 4. On **jette** les ratings 5-6 (ambigus).


In [ ]:
def to_label(r):
    if r >= 7:
        return 'positif'
    if r <= 4:
        return 'negatif'
    return None

df_small['label'] = df_small['rating'].apply(to_label)
df_bin = df_small.dropna(subset=['label']).reset_index(drop=True)
print(f'Avis gardés : {len(df_bin)} / {len(df_small)}')
print(df_bin['label'].value_counts(normalize=True).rename('proportion'))


## 5. À toi

À partir d'ici, suis le sujet (`TP_Note_Sujets.md` § TP-E) :

1. EDA texte : longueur des avis, distribution des ratings, mots les plus fréquents (avec/sans stop-words).
2. Train/test split stratifié sur `label`. **Fit le vectorizer sur le train uniquement.**
3. Compare CountVectorizer vs TF-IDF — chiffres à l'appui.
4. Au moins 2 modèles : LogReg, Naive Bayes (Multinomial). Optionnel : SVM linéaire.
5. Évaluation : accuracy, F1, matrice de confusion. Affiche les **20 mots les plus prédictifs** par classe.
6. Analyse critique : 3 exemples mal classés, biais de domaine (vocabulaire médical).
7. Bonus optionnel : pipeline Hugging Face `sentiment-analysis`.
